# Brand Logo Detection (YOLOv8)

Prepare LogoDet-3K (or compatible) data, train the YOLOv8 logo detector, and verify inference with the brand API model.


In [19]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for cand in [repo_root, *repo_root.parents]:
    if (cand / 'app').exists():
        repo_root = cand
        break
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)


Repo root: /Users/pratik_n/Desktop/MyComputer/Sentifargo


## Data requirements

Place LogoDet-3K (or a compatible dataset) under data/raw/brand/LogoDet-3K or data/raw/brand/logodet3k. The preparation script will convert it into data/processed/brand_yolo.


In [20]:
# Update paths if your data lives elsewhere.



In [21]:
raw_dir = repo_root / 'data' / 'raw' / 'brand'
print('Raw brand dir:', raw_dir, 'exists=', raw_dir.exists())


Raw brand dir: /Users/pratik_n/Desktop/MyComputer/Sentifargo/data/raw/brand exists= True


## Prepare dataset into YOLO format


In [22]:
# !python scripts/prepare_brand_data.py


## Train YOLOv8 detector


In [23]:
# Quick test (adjust epochs/imgsz/batch as needed).
# You can set BRAND_TRAIN_MAX_IMAGES to limit images for a smoke test.

# Example:
# BRAND_YOLO_MODEL=yolov8n.pt BRAND_EPOCHS=10 BRAND_IMGSZ=320 BRAND_BATCH=16 \
# BRAND_DEVICE=auto BRAND_TRAIN_MAX_IMAGES=5000 BRAND_VAL=true \
# python -m src.train.train_brand_logo_detector


## Run a local inference sample


In [24]:
from pathlib import Path
from src.vision.brand.recognizer import predict_image_bytes

sample = None
for cand in (repo_root / 'data' / 'processed' / 'brand_yolo').rglob('*.jpg'):
    sample = cand
    break

if sample is None:
    print('No sample image found under data/processed/brand_yolo')
else:
    detections = predict_image_bytes(sample.read_bytes(), conf=0.25)
    print('Detections:', detections[:5])


RuntimeError: Brand model not trained. Run scripts/prepare_brand_data.py then python -m src.train.train_brand_logo_detector to create artifacts/brand/yolo_logo_det.pt

In [ ]:
# METRICS_EXPORT_BRAND_YOLO
import pandas as pd
from pathlib import Path

candidates = [
    repo_root / 'models' / 'brand' / 'full' / 'results.csv',
    repo_root / 'models' / 'brand' / 'fast_run_1epoch' / 'results.csv',
    repo_root / 'runs' / 'detect' / 'train8' / 'results.csv',
    repo_root / 'runs' / 'detect' / 'train9' / 'results.csv',
    repo_root / 'runs' / 'detect' / 'train13' / 'results.csv',
]
best = None
for path in candidates:
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if 'metrics/mAP50-95(B)' not in df.columns:
        continue
    row = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
    score = float(row['metrics/mAP50-95(B)'])
    if best is None or score > best['mAP50-95']:
        best = {
            'mAP50-95': score,
            'mAP50': float(row.get('metrics/mAP50(B)', 0.0)),
            'precision': float(row.get('metrics/precision(B)', 0.0)),
            'recall': float(row.get('metrics/recall(B)', 0.0)),
            'source': str(path),
        }

rows = []
if best:
    rows = [
        {'Metric': 'mAP50', 'mean': best['mAP50'], 'std': ''},
        {'Metric': 'mAP50-95', 'mean': best['mAP50-95'], 'std': ''},
        {'Metric': 'precision', 'mean': best['precision'], 'std': ''},
        {'Metric': 'recall', 'mean': best['recall'], 'std': ''},
    ]
report_path = repo_root / 'reports' / 'metrics_brand_yolo.csv'
df_new = pd.DataFrame(rows)
if report_path.exists():
    df_old = pd.read_csv(report_path)
    if 'std' not in df_old.columns:
        df_old['std'] = ''
    df_old = df_old[~df_old['Metric'].isin(df_new['Metric'])]
    df_out = pd.concat([df_old, df_new], ignore_index=True)
else:
    df_out = df_new
report_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(report_path, index=False)
print('Saved metrics to', report_path)
